# Сравнение экспериментов

Читает `metrics.csv` и `summary.json` из всех прогонов в `experiments/`,  
строит таблицу и графики для выбора лучшей конфигурации.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

EXPERIMENTS_DIR = Path('../experiments')
ZEROSHOT_DIR = EXPERIMENTS_DIR / 'zeroshot'

In [ ]:
def load_summary(run_dir: Path) -> dict | None:
    """Читает summary.json из директории прогона."""
    p = run_dir / 'summary.json'
    if not p.exists():
        return None
    with open(p) as f:
        return {**json.load(f), 'run': run_dir.name, 'run_path': str(run_dir)}

def load_metrics_csv(run_dir: Path) -> pd.DataFrame | None:
    """Читает metrics.csv из директории прогона."""
    p = run_dir / 'metrics.csv'
    if not p.exists():
        return None
    df = pd.read_csv(p)
    df['run'] = run_dir.name
    return df

def load_eval_metrics(run_dir: Path) -> dict | None:
    """Читает eval_results/metrics.json если он есть."""
    p = run_dir / 'eval_results' / 'metrics.json'
    if not p.exists():
        return None
    with open(p) as f:
        return json.load(f)

# Собираем все прогоны
runs = []
for date_dir in sorted(EXPERIMENTS_DIR.glob('????-??-??')):
    for ts_dir in sorted(date_dir.iterdir()):
        for run_dir in sorted(ts_dir.iterdir()):
            s = load_summary(run_dir)
            if s:
                s['eval'] = load_eval_metrics(run_dir)
                runs.append(s)

# Zero-shot
for zs_dir in sorted(ZEROSHOT_DIR.glob('*')):
    p = zs_dir / 'metrics.json'
    if p.exists():
        with open(p) as f:
            runs.append({'run': f'zeroshot/{zs_dir.name}', 'eval': json.load(f)})

print(f'Найдено прогонов: {len(runs)}')

In [ ]:
# Таблица: итоговые метрики по всем прогонам
rows = []
for r in runs:
    row = {'run': r['run']}
    if r.get('final_train_loss'):
        row['train_loss'] = round(r['final_train_loss'], 4)
    if r.get('eval'):
        for mode in ['image', 'txt', 'multimodal', 'all']:
            if mode in r['eval']:
                m = r['eval'][mode]
                row[f'{mode}/r@10']   = round(m.get('recall@10', 0), 3)
                row[f'{mode}/p@10']   = round(m.get('precision@10', 0), 3)
                row[f'{mode}/mrr']    = round(m.get('mrr', 0), 3)
    rows.append(row)

df_summary = pd.DataFrame(rows).set_index('run')
df_summary.style.highlight_max(axis=0, props='background-color: #c8f7c5')

In [ ]:
# Loss curves
all_metrics = []
for date_dir in sorted(EXPERIMENTS_DIR.glob('????-??-??')):
    for ts_dir in sorted(date_dir.iterdir()):
        for run_dir in sorted(ts_dir.iterdir()):
            df = load_metrics_csv(run_dir)
            if df is not None:
                all_metrics.append(df)

if all_metrics:
    df_all = pd.concat(all_metrics, ignore_index=True)
    train_df = df_all[df_all['train_loss'].notna()] if 'train_loss' in df_all.columns else pd.DataFrame()

    if not train_df.empty:
        fig, ax = plt.subplots(figsize=(10, 4))
        for run, grp in train_df.groupby('run'):
            ax.plot(grp['step'], grp['train_loss'], label=run, linewidth=1.2)
        ax.set_xlabel('step')
        ax.set_ylabel('train loss')
        ax.set_title('Training loss curves')
        ax.legend(fontsize=8)
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
        plt.tight_layout()
        plt.show()
else:
    print('Нет прогонов с metrics.csv')

In [ ]:
# Bar chart: Recall@10 по режимам для всех прогонов
modes = ['image', 'txt', 'multimodal']
r10_data = {r['run']: {m: r['eval'][m].get('recall@10', 0) for m in modes if r.get('eval') and m in r['eval']} for r in runs}
r10_df = pd.DataFrame(r10_data).T

if not r10_df.empty:
    r10_df.plot(kind='bar', figsize=(max(6, len(r10_df)*1.5), 4), ylim=(0, 1))
    plt.title('Recall@10 по режимам')
    plt.ylabel('Recall@10')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('Нет данных eval')